# Subjective Well-Being — The One Notebook That Should Be Simulated

There is no free U.S. subnational life-satisfaction series. The archived D30 notebook in this
repository handled that by generating seven independent normal variates, building a target
from three of them, assigning a region at random — and then reporting regression
coefficients, random-forest feature importances, cluster profiles and a regional ranking as
findings about human happiness.

This rebuild does not replace that data with real data, because there is none to use. It does
something more useful: it **declares the process that made the numbers**, regenerates the
archived dataset exactly, and scores every claim the archived notebook published against a
truth that is known rather than estimated.

All ten published numbers reproduce. Here is what they were worth.

Of the three determinants the archive reported, **one — `unemployment_rate`, at p = 0.0482 —
has a true coefficient of exactly zero**. A second, `health_coverage_pct`, is real and was
overstated 1.82×. The third, `median_income`, is real, was estimated well, and was **displayed
as `0.0000`** because its true value is 2 × 10⁻⁵ and the print format carried four decimals.
The random forest assigned **39.90%** of its feature importance to the two predictors whose
true effect is zero. The reported test R² of 0.6144 sits at the **98th percentile** of what
that same model on that same data produces across 300 different train/test splits. And the
three clusters the archive named *High*, *Moderate* and *Lower Well-Being* are separated by
1.06 points — the **94th percentile of what the identical procedure produces on data with no
signal in it at all**.

Then the harder question, and the one that makes this a lesson rather than an autopsy: was
any of that unusual? Across 2,000 independently drawn datasets, **no**. The false-positive
rate is 8.60% against a nominal 9.75%. The archive was not unlucky and its pipeline is not
broken. It ran a conventional analysis, at a conventional threshold, on 50 units, and
published what came out — and nothing in what came out distinguished the true findings from
the false one.

This is the corpus's only artifact with `data_mode: "illustrative"`, and the only one where
randomness is permitted. The reason is exact: **you cannot demonstrate false discovery on
real data, because on real data you do not know which discoveries are false.**

## Learning Objectives

1. **Score an analysis rather than read it**, by working where the data-generating process is
   declared and the true coefficients are known by construction.
2. **Separate a bad result from a bad method.** Measure how often a design produces a false
   finding before concluding that a particular false finding means something.
3. **Read a coefficient's standard error as the finding**, and see two real effects at the
   same n, in the same model, recovered with completely different reliability.
4. **Recognise a performance figure that is a property of the split**, not of the model.
5. **Recognise procedures that cannot return nothing** — clustering, importance ranking,
   "highest region" — and know what they produce when given no signal.
6. **Distinguish the two errors this design makes**, and see that the notebook's output shows
   neither of them.

## Prerequisites

- Handbook §1.1 (ACS margins of error) for the contrast: there the publisher states the
  uncertainty and the analyst may ignore it. Here nobody states it and the analyst cannot
  recover it from the output at all.
- Handbook §7.3 (precision and rank) for the general result this notebook meets from the
  other direction.
- OLS, `train_test_split`, k-means and random-forest feature importance as procedures. No
  prior knowledge of their failure modes is assumed — that is the notebook.

## Data Sources

**None.** This notebook retrieves nothing.

Its data comes from `krl_tutorials_data.simulation.simulate_wellbeing`, the only synthetic
generator in this package. That function is a faithful re-implementation of the process
embedded in the archived notebook at
`notebooks/09_planned/D30_subjective_wellbeing/D30_D30_subjective_wellbeing.ipynb` —
including its legacy `RandomState` draw order and one draw the archive discarded, both kept
because removing either changes every number downstream and the reproduction in section 2 is
what the rest of the notebook rests on.

The generator returns a `SimulationSpec`, not a `DataProvenance`. The two are deliberately
different types: a spec has no endpoint, no vintage and no publisher, because nothing is
upstream of it. `tests/test_simulation.py` pins the archived numbers, verifies that the
declared coefficients are the ones actually used, and enforces that no module in the
retrieval path imports this one.

**Why an original source could not be used.** BRFSS publishes poor-mental-health-days, which
is a symptom-frequency measure and not life satisfaction. The Gallup-Sharecare index is not
freely redistributable. The World Happiness Report is national. Substituting any of these
would produce a notebook about a different quantity while keeping the old title — which is a
milder version of exactly the failure this notebook exists to document.

## 1. The process, declared

Everything below depends on this being written down first. A simulation whose parameters are
buried in the code that uses them is not a yardstick; it is another analysis.

Two features of the generator matter later.

`median_income` enters life satisfaction at **2 × 10⁻⁵ points per dollar** — 0.2 points per
$10,000. It is a real effect with a very small number attached to it, because of the units.

`region` and `poor_mental_health_days` enter nothing at all. They are independent draws. Any
regional difference, and any correlation with mental-health days, recovered from this frame
is an artifact with certainty, not with suspicion.

In [1]:
"""Configuration."""
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.formula.api import ols

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.simulation import (
    WELLBEING_NULL_PREDICTORS,
    WELLBEING_PREDICTORS,
    WELLBEING_TRUTH,
    simulate_wellbeing,
)

#: The seed the archived notebook used. Everything in section 2 is that
#: notebook's own output, regenerated.
ARCHIVE_SEED = 42

#: The test R-squared printed in the archived notebook's own stored output.
#: Transcribed from the forensic corpus -- it is the claim under test, not
#: something this notebook computes.
ARCHIVE_TEST_R2 = 0.6136

#: How many independent datasets each sweep draws. Large enough that a rate
#: quoted to one decimal place is stable; small enough to execute in a tutorial.
RUNS = 2_000
SLOW_RUNS = 600

#: The conventional threshold the archived notebook used to decide what to report.
ALPHA = 0.05

#: The estimand, named once. Every model below regresses or predicts this and
#: nothing else, and no cell selects it by column position.
TARGET = "life_satisfaction"

#: The archived pipeline's own settings, reproduced rather than improved.
TEST_SIZE = 0.2
CLUSTER_FEATURES = [TARGET, "poor_mental_health_days", "happiness_score", "median_income"]
FORMULA = f"{TARGET} ~ " + " + ".join(WELLBEING_PREDICTORS)

TRUTH = WELLBEING_TRUTH.as_dict()
NULLS = set(WELLBEING_NULL_PREDICTORS)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 110)


def fit_ols(frame: pd.DataFrame):
    """The archived regression, unchanged."""
    return ols(FORMULA, data=frame).fit()


print(f"generator      : archived D30 well-being process")
print(f"predictors     : {', '.join(WELLBEING_PREDICTORS)}")
print(f"true effects   : " + ", ".join(f"{k}={v:g}" for k, v in TRUTH.items()))
print(f"null predictors: {', '.join(sorted(NULLS))}  (true coefficient exactly zero)")


generator      : archived D30 well-being process
predictors     : median_income, unemployment_rate, health_coverage_pct, education_bachelors_pct
true effects   : median_income=2e-05, health_coverage_pct=0.02, unemployment_rate=0, education_bachelors_pct=0
null predictors: education_bachelors_pct, unemployment_rate  (true coefficient exactly zero)


In [2]:
"""Regenerate the archived dataset and state what is true of it."""
archive, spec = simulate_wellbeing(seed=ARCHIVE_SEED)

print(spec)
print()
for note in spec.notes:
    print(f"  - {note}")
print()
# A constant target yields a perfect-looking fit and a meaningless one. In a
# simulation this is nearly free to check and there is no excuse for skipping it.
target_variance = float(archive[TARGET].var())
assert target_variance > 0, f"{TARGET} is constant; nothing below would mean anything"
assert archive[TARGET].nunique() == len(archive), (
    f"{TARGET} has ties, which the declared continuous process should not produce"
)

print(f"shape   : {archive.shape}")
print(f"target  : {TARGET}, variance {target_variance:.4f}, "
      f"{archive[TARGET].nunique()} distinct values in {len(archive)} rows")
print(archive[list(WELLBEING_PREDICTORS) + [TARGET]].describe().round(2).to_string())


SIMULATED -- archived D30 well-being process, seed=42, n=50, signal=1, spec 1ed714060dc2

  - SIMULATED. Nothing here was measured. This frame exists so that a published analysis of it can be scored against a truth that is known rather than estimated.
  - True coefficients on life_satisfaction: median_income=2e-05, health_coverage_pct=0.02, unemployment_rate=0, education_bachelors_pct=0. The two at exactly zero are unemployment_rate, education_bachelors_pct.
  - region is an independent categorical draw and enters no outcome. Any regional difference recovered from this frame is an artifact.
  - poor_mental_health_days is an independent draw and enters no outcome.
  - happiness_score = 0.8 * life_satisfaction + N(0, 0.5), so it carries no information the target does not already have.
  - Values are clipped as in the archived process, which makes the true relationship slightly non-linear near the boundaries.

shape   : (50, 9)
target  : life_satisfaction, variance 0.3145, 50 distinct val

## 2. Reproduction

Before scoring the archived notebook's claims, establish that we are scoring *its* claims and
not a paraphrase. Each row below is a number the archived notebook printed, checked against
the regenerated data.

In [3]:
"""Does the regenerated data reproduce what the archived notebook published?"""
#: Verbatim from the stored outputs of the archived notebook.
ARCHIVED_OLS = {
    "unemployment_rate": (0.0770, 0.0482),
    "health_coverage_pct": (0.0365, 0.0051),
    "median_income": (0.0000, 0.0000),
}
ARCHIVED_MEANS = {
    "life_satisfaction": 7.37, "happiness_score": 5.99, "poor_mental_health_days": 3.52,
}

model = fit_ols(archive)
rows = []
for name, (beta, pval) in ARCHIVED_OLS.items():
    rows.append({
        "claim": f"{name} coefficient",
        "archive": f"{beta:.4f}",
        "reproduced": f"{model.params[name]:.4f}",
        "agrees": round(float(model.params[name]), 4) == beta,
    })
    rows.append({
        "claim": f"{name} p-value",
        "archive": f"{pval:.4f}",
        "reproduced": f"{model.pvalues[name]:.4f}",
        "agrees": round(float(model.pvalues[name]), 4) == pval,
    })
for column, mean in ARCHIVED_MEANS.items():
    rows.append({
        "claim": f"mean {column}",
        "archive": f"{mean:.2f}",
        "reproduced": f"{archive[column].mean():.2f}",
        "agrees": round(float(archive[column].mean()), 2) == mean,
    })
regional = archive.groupby("region")[TARGET].mean()
rows.append({
    "claim": "highest-satisfaction region",
    "archive": "Northeast 7.55",
    "reproduced": f"{regional.idxmax()} {regional.max():.2f}",
    "agrees": regional.idxmax() == "Northeast" and round(float(regional.max()), 2) == 7.55,
})

check = pd.DataFrame(rows)
print(check.to_string(index=False))
print()
print(f"claims reproduced: {check['agrees'].sum()} of {len(check)}")


                          claim        archive     reproduced  agrees
  unemployment_rate coefficient         0.0770         0.0770    True
      unemployment_rate p-value         0.0482         0.0482    True
health_coverage_pct coefficient         0.0365         0.0365    True
    health_coverage_pct p-value         0.0051         0.0051    True
      median_income coefficient         0.0000         0.0000    True
          median_income p-value         0.0000         0.0000    True
         mean life_satisfaction           7.37           7.37    True
           mean happiness_score           5.99           5.99    True
   mean poor_mental_health_days           3.52           3.52    True
    highest-satisfaction region Northeast 7.55 Northeast 7.55    True

claims reproduced: 10 of 10


**Ten of ten.** The coefficients, the p-values to four decimals, the sample
means, and the regional winner all come back. What follows is an audit of the archived
artifact itself, not of something like it.

That fidelity cost something worth naming. The generator uses `np.random.RandomState` rather
than the modern `default_rng`, in the archive's draw order, and it retains a `normal(6.8, 0.7,
50)` draw that the archive computed and then overwrote two statements later. Deleting that
dead line advances the bit stream and changes every number after it. A test asserts that it
does, so nobody tidies it away.

## 3. Scoring the published determinants

The archived notebook printed a section headed **KEY DETERMINANTS (OLS Regression)** listing
three variables with coefficients and p-values. Here is each one against the coefficient that
actually made the data.

In [4]:
"""Score each published determinant against the coefficient that made the data."""
scored = []
for name in WELLBEING_PREDICTORS:
    estimate = float(model.params[name])
    true = TRUTH[name]
    reported = float(model.pvalues[name]) < ALPHA
    if true == 0:
        verdict = "FALSE POSITIVE" if reported else "correctly silent"
    elif not reported:
        verdict = "missed"
    elif np.sign(estimate) != np.sign(true):
        verdict = "reported with the WRONG SIGN"
    else:
        verdict = f"true, overstated {estimate / true:.2f}x"
    scored.append({
        "predictor": name,
        "true": f"{true:g}",
        "estimate": f"{estimate:.6g}",
        "std error": f"{float(model.bse[name]):.6g}",
        "t": f"{float(model.params[name]) / float(model.bse[name]):.2f}",
        "p": f"{float(model.pvalues[name]):.4f}",
        "reported?": "yes" if reported else "no",
        "verdict": verdict,
    })

print(pd.DataFrame(scored).to_string(index=False))
print()
published = [s for s in scored if s["reported?"] == "yes"]
false_ones = [s for s in published if s["verdict"] == "FALSE POSITIVE"]
print(f"the archive published {len(published)} determinants; {len(false_ones)} of them "
      f"has a true coefficient of exactly zero: "
      f"{', '.join(s['predictor'] for s in false_ones) or 'none'}")
print(f"the archive displayed the median_income coefficient as "
      f"{float(model.params['median_income']):.4f} at four decimal places; "
      f"it is {float(model.params['median_income']):.6g}, against a true {TRUTH['median_income']:g}")


              predictor  true    estimate   std error     t      p reported?                verdict
          median_income 2e-05 2.73053e-05 4.72064e-06  5.78 0.0000       yes true, overstated 1.37x
      unemployment_rate     0   0.0769697   0.0378955  2.03 0.0482       yes         FALSE POSITIVE
    health_coverage_pct  0.02   0.0364663   0.0123788  2.95 0.0051       yes true, overstated 1.82x
education_bachelors_pct     0  -0.0179874  0.00984571 -1.83 0.0743        no       correctly silent

the archive published 3 determinants; 1 of them has a true coefficient of exactly zero: unemployment_rate
the archive displayed the median_income coefficient as 0.0000 at four decimal places; it is 2.73053e-05, against a true 2e-05


Three findings, and each fails differently.

**`unemployment_rate`: +0.0770, p = 0.0482.** The true coefficient is **exactly zero**. This
is a false positive, and it is not a near miss — it was published, with a positive sign,
implying that higher unemployment raises life satisfaction.

**`health_coverage_pct`: 0.0365, p = 0.0051.** Real, and **overstated 1.82×** against a true
0.02. The p-value is the smallest of the three after income, which is to say the archive was
most confident about the estimate that was furthest from the truth in relative terms.

**`median_income`: displayed as `0.0000`, p = 0.0000.** This is the archive's best estimate.
The true value is 2 × 10⁻⁵, the estimate is 2.73 × 10⁻⁵, the sign is right and the p-value is
vanishing. And a four-decimal print format rendered it **indistinguishable from zero**. The
one coefficient the analysis got right is the one a reader cannot see.

**`education_bachelors_pct`: p = 0.0743, not reported.** Correct — its true effect is zero.
It is also the narrowest escape in the table; at α = 0.10 the archive would have published two
false determinants instead of one.

## 4. Was that unusual?

Everything in section 3 describes one dataset. The interesting question is whether the archive
was unlucky or whether the design produces this.

Draw 2,000 independent datasets from the same declared process and run the same regression on
each.

In [5]:
"""Was that bad luck, or is this design prone to it? Draw many datasets."""
records = []
for seed in range(RUNS):
    frame, _ = simulate_wellbeing(seed=seed)
    fit = fit_ols(frame)
    regional_means = frame.groupby("region")[TARGET].mean()
    records.append({
        "seed": seed,
        **{f"p::{k}": float(fit.pvalues[k]) for k in WELLBEING_PREDICTORS},
        **{f"b::{k}": float(fit.params[k]) for k in WELLBEING_PREDICTORS},
        "regional gap": float(regional_means.max() - regional_means.min()),
    })
sweep = pd.DataFrame(records)

summary = []
for name in WELLBEING_PREDICTORS:
    significant = sweep[f"p::{name}"] < ALPHA
    summary.append({
        "predictor": name,
        "true": f"{TRUTH[name]:g}",
        f"reported at p<{ALPHA}": f"{significant.mean():.2%}",
        "positive when reported": f"{int((significant & (sweep[f'b::{name}'] > 0)).sum())}"
                                  f"/{int(significant.sum())}",
    })
print(pd.DataFrame(summary).to_string(index=False))

any_null = (sweep[[f"p::{k}" for k in sorted(NULLS)]] < ALPHA).any(axis=1)
nominal = 1 - (1 - ALPHA) ** len(NULLS)
print()
print(f"at least one zero-effect predictor reported: {any_null.mean():.2%} of {RUNS:,} runs")
print(f"nominal rate for {len(NULLS)} independent tests at alpha={ALPHA}: {nominal:.2%}")


              predictor  true reported at p<0.05 positive when reported
          median_income 2e-05             97.90%              1958/1958
      unemployment_rate     0              4.55%                  45/91
    health_coverage_pct  0.02             25.60%                512/512
education_bachelors_pct     0              4.50%                  45/90

at least one zero-effect predictor reported: 8.60% of 2,000 runs
nominal rate for 2 independent tests at alpha=0.05: 9.75%


The design is not broken. It is behaving exactly as advertised.

Each zero-effect predictor is reported at p < 0.05 in about **4.5%** of runs, against a
nominal 5%. At least one of the two is reported in **8.60%** of runs, against a nominal
**9.75%**. The archive drew one of them.

That is the uncomfortable part. There was no methodological error to catch. A conventional
regression, a conventional threshold, one dataset, and roughly a one-in-twelve chance of
publishing a determinant that does not exist — and the notebook's output offered a reader
nothing at all with which to tell.

Look also at the column the eye skips. **`health_coverage_pct` is a real effect and it is
reported in only 25.6% of runs.** Three times in four, this design misses it entirely. So the
design makes both errors at once: it publishes a nonexistent effect about one run in twelve,
and it fails to find a genuine one about three runs in four. Neither is visible in a single
notebook's output.

## 5. Two real effects, two different worlds

`median_income` and `health_coverage_pct` are both genuine. They are estimated from the same
50 rows, by the same model, in the same regression. They are not comparable.

In [6]:
"""The two real effects: how reliably does n=50 recover them?"""
recovery = []
for name in ("median_income", "health_coverage_pct"):
    estimates = sweep[f"b::{name}"]
    true = TRUTH[name]
    ratio = estimates / true
    recovery.append({
        "predictor": name,
        "true": f"{true:g}",
        "median estimate": f"{estimates.median():.6g}",
        "median ratio": f"{ratio.median():.2f}x",
        "5th-95th ratio": f"[{np.percentile(ratio, 5):.2f}x, {np.percentile(ratio, 95):.2f}x]",
        "wrong sign": f"{(np.sign(estimates) != np.sign(true)).mean():.2%}",
        "reported": f"{(sweep[f'p::{name}'] < ALPHA).mean():.1%}",
    })
print(pd.DataFrame(recovery).to_string(index=False))
print()
print("At the archive's own seed:")
for name in ("median_income", "health_coverage_pct"):
    est, true = float(model.params[name]), TRUTH[name]
    pct = float((sweep[f"b::{name}"] < est).mean())
    print(f"  {name:<22} {est:.6g} = {est / true:.2f}x true, "
          f"the {100 * pct:.0f}th percentile of what this design produces")


          predictor  true median estimate median ratio  5th-95th ratio wrong sign reported
      median_income 2e-05     2.00231e-05        1.00x  [0.61x, 1.41x]      0.00%    97.9%
health_coverage_pct  0.02       0.0198673        0.99x [-0.21x, 2.21x]      8.55%    25.6%

At the archive's own seed:
  median_income          2.73053e-05 = 1.37x true, the 93th percentile of what this design produces
  health_coverage_pct    0.0364663 = 1.82x true, the 87th percentile of what this design produces


`median_income` is recovered at a median of **1.00×** with a 5th–95th range of
**0.61× to 1.41×**, is never wrong-signed in 2,000 runs, and is detected **97.9%** of the
time.

`health_coverage_pct` is recovered at a median of **0.99×** — unbiased, in the median — with a
5th–95th range of **−0.21× to 2.21×**. It comes out **wrong-signed in 8.55%** of runs. It is
detected 25.6% of the time.

Both are unbiased. Only one is *useful*, and the thing that separates them is in the table in
section 3, in the column the archive did not print: the standard error. The archive's own fit
gives `median_income` a coefficient **5.78** standard errors from zero and
`health_coverage_pct` one **2.95** — the `t` column in section 3.

The archive's two overstatements — 1.37× and 1.82× — are the **93rd and 87th percentiles** of
what this design produces. Unremarkable draws from distributions it never displayed.

**[rule]** A coefficient without its standard error is not a finding, and a table of
coefficients sorted by p-value is not a ranking of how much you know.

And then the limit of that rule, which section 3's table shows plainly. The false positive
sits at **t = 2.03** and the real-but-fragile effect at **t = 2.95** — nine-tenths of a
standard error apart, both past the threshold, both published. The `t` column separates the
one estimate worth trusting from the other two. It does **not** separate the true ones from
the false one, and nothing computed from a single dataset does. That is what sections 4 and
11 are for: the question "is this finding real?" has no answer inside one run, and the
question "how often is a finding like this false?" has one.

## 6. What would it have taken?

The coverage effect is real and n = 50 cannot see it reliably. This is answerable rather than
lamentable: run the same estimate at increasing sample sizes.

In [7]:
"""What sample size would the coverage effect have needed?"""
name = "health_coverage_pct"
true = TRUTH[name]
sizes = []
for n in (50, 100, 250, 500, 1000):
    estimates = np.array([
        float(fit_ols(simulate_wellbeing(seed=s, n=n)[0]).params[name])
        for s in range(300)
    ])
    ratio = estimates / true
    sizes.append({
        "n": n,
        "wrong sign": f"{(estimates < 0).mean():.2%}",
        "5th-95th ratio": f"[{np.percentile(ratio, 5):.2f}x, {np.percentile(ratio, 95):.2f}x]",
        "within +/-25% of true": f"{(np.abs(ratio - 1) < 0.25).mean():.1%}",
    })
print(pd.DataFrame(sizes).to_string(index=False))


   n wrong sign  5th-95th ratio within +/-25% of true
  50      9.33% [-0.18x, 2.19x]                 25.0%
 100      2.00%  [0.14x, 1.95x]                 38.3%
 250      0.33%  [0.51x, 1.52x]                 56.7%
 500      0.00%  [0.65x, 1.36x]                 70.7%
1000      0.00%  [0.75x, 1.27x]                 87.3%


At **n = 50** the estimate is wrong-signed 9.33% of the time and lands within
±25% of the truth in **25.0%** of runs. The sign stops being a coin at **n ≈ 250**
(0.33% wrong) and is never wrong by **n = 500**. Landing inside ±25% is slower: **70.7%**
of runs at n = 500 and **87.3%** at n = 1,000.

The archived notebook had fifty units because the United States has fifty states. That is not
a sample size chosen for a question; it is a sample size inherited from a map. Anything about
state-level variation in a modest effect is, at that n, a coin weighted slightly better than
even — and no amount of care in the modelling changes it.

## 7. Feature importance, against coefficients known to be zero

The archive reported a random-forest importance ranking and named its top feature. Importance
rankings are attractive because they always produce an ordering, always sum to one, and never
report that a variable does not matter.

In [8]:
"""Random-forest feature importance, against coefficients known to be zero."""
features = list(WELLBEING_PREDICTORS)
X_train, X_test, y_train, y_test = train_test_split(
    archive[features], archive[TARGET],
    test_size=TEST_SIZE, random_state=ARCHIVE_SEED,
)
forest = RandomForestRegressor(
    n_estimators=100, max_depth=5, random_state=ARCHIVE_SEED
).fit(X_train, y_train)

ranked = pd.DataFrame({
    "feature": features,
    "importance": forest.feature_importances_,
    "true effect": [TRUTH[f] for f in features],
}).sort_values("importance", ascending=False)
ranked["is null"] = ranked["feature"].isin(NULLS)
print(ranked.to_string(index=False, formatters={
    "importance": "{:.4f}".format, "true effect": "{:g}".format}))
print()
null_share = ranked.loc[ranked["is null"], "importance"].sum()
print(f"importance assigned to the two predictors whose true effect is exactly zero: "
      f"{null_share:.2%}")
#: The archive's stored output says 0.6136. This pipeline reproduces 0.6144 on the
#: same seed, so the two are close and NOT identical. An earlier revision printed
#: the reproduced figure under the label "as the archive reported it", which told
#: a reader the reproduction was exact when it is not.
reproduced_r2 = r2_score(y_test, forest.predict(X_test))
print(f"test R2 the archive published : {ARCHIVE_TEST_R2:.4f}")
print(f"test R2 reproduced here       : {reproduced_r2:.4f} on n_test = {len(y_test)}")
print(f"difference                    : {reproduced_r2 - ARCHIVE_TEST_R2:+.4f}")


                feature importance true effect  is null
          median_income     0.3162       2e-05    False
    health_coverage_pct     0.2848        0.02    False
education_bachelors_pct     0.2037           0     True
      unemployment_rate     0.1953           0     True

importance assigned to the two predictors whose true effect is exactly zero: 39.90%
test R2 the archive published : 0.6136
test R2 reproduced here       : 0.6144 on n_test = 10
difference                    : +0.0008


The ranking gets the top two right — and then assigns **39.90%** of total
importance to the two predictors whose true effect is **exactly zero**. `education_bachelors_pct`
receives 20.37%, more than a fifth of the model's explanatory credit, for contributing nothing
to the target by construction.

That is not a bug in the method. A random forest splits on whatever reduces variance in the
training sample, a null predictor reduces some by chance, and the importances are normalised
to sum to one whether or not anything matters. **The procedure has no way to output "this
variable is irrelevant", so it never does.**

**[rule]** Importance shares are relative within the feature set you supplied. They do not
establish that any member of that set is related to the target.

## 8. The reported R² is a property of the split

The archive reported **Test R²: 0.6136** — a respectable figure that a reader would take as a
statement about the model. Hold the fifty rows fixed, hold the model fixed, and vary only
which ten rows land in the test set.

In [9]:
"""Hold the data fixed. Vary only which ten rows land in the test set."""
splits = []
for state in range(300):
    X_a, X_b, y_a, y_b = train_test_split(
        archive[features], archive[TARGET],
        test_size=TEST_SIZE, random_state=state,
    )
    fitted = RandomForestRegressor(
        n_estimators=100, max_depth=5, random_state=ARCHIVE_SEED
    ).fit(X_a, y_a)
    splits.append(r2_score(y_b, fitted.predict(X_b)))
splits = np.array(splits)

archive_r2 = ARCHIVE_TEST_R2
print(f"same 50 rows, same model, {len(splits)} different values of random_state:")
print(f"  median R2 : {np.median(splits):.3f}")
print(f"  5th-95th  : [{np.percentile(splits, 5):.3f}, {np.percentile(splits, 95):.3f}]")
print(f"  minimum   : {splits.min():.3f}")
print(f"  maximum   : {splits.max():.3f}")
print()
print(f"the archive published {archive_r2:.4f}, at random_state={ARCHIVE_SEED}")
print(f"that is the {100 * (splits < archive_r2).mean():.0f}th percentile of this distribution")


same 50 rows, same model, 300 different values of random_state:
  median R2 : 0.202
  5th-95th  : [-0.570, 0.536]
  minimum   : -1.421
  maximum   : 0.673

the archive published 0.6136, at random_state=42
that is the 98th percentile of this distribution


Same data. Same model. Same everything except an integer passed to
`train_test_split`.

The median R² is **0.202**. The 5th–95th range is **−0.570 to 0.536**. The minimum is
**−1.421** — a model performing worse than predicting the test-set mean. The archive's
**0.6144** is the **98th percentile** of this distribution, and higher than the maximum of the
5th–95th band.

A ten-row test set cannot measure a model. It can only report which ten rows it got. The
archive published the best of roughly three hundred available answers and did not know it was
choosing.

## 9. Clusters, and what noise looks like

The archive ran k-means at k = 3 and labelled the results **High**, **Moderate** and **Lower
Well-Being**. Reproduce the table, then generate the comparison the archive never had: the
same procedure on data where every true effect has been severed.

In [10]:
"""The cluster profile the archive named, and what pure noise produces."""
scaled = StandardScaler().fit_transform(archive[CLUSTER_FEATURES])
labels = KMeans(n_clusters=3, random_state=ARCHIVE_SEED, n_init=10).fit_predict(scaled)
profile = archive.assign(cluster=labels).groupby("cluster").agg(
    n=(TARGET, "size"),
    life_satisfaction=(TARGET, "mean"),
    median_income=("median_income", "mean"),
)
profile["archive label"] = ["High Well-Being", "Moderate Well-Being", "Lower Well-Being"]
print(profile.to_string(formatters={
    "n": "{:,.0f}".format, "life_satisfaction": "{:.2f}".format,
    "median_income": "${:,.0f}".format}))

observed_spread = float(
    profile[TARGET].max() - profile[TARGET].min()
)
print()
print(f"spread between the highest and lowest cluster: {observed_spread:.2f} points")

noise_spreads = []
for seed in range(SLOW_RUNS):
    frame, _ = simulate_wellbeing(seed=seed, signal=0.0)
    noise_scaled = StandardScaler().fit_transform(frame[CLUSTER_FEATURES])
    noise_labels = KMeans(
        n_clusters=3, random_state=ARCHIVE_SEED, n_init=10
    ).fit_predict(noise_scaled)
    means = frame.assign(cluster=noise_labels).groupby("cluster")[TARGET].mean()
    noise_spreads.append(float(means.max() - means.min()))
noise_spreads = np.array(noise_spreads)

print(f"the same procedure on data where every true effect is zero, {SLOW_RUNS} runs:")
print(f"  median spread : {np.median(noise_spreads):.2f} points")
print(f"  5th-95th      : [{np.percentile(noise_spreads, 5):.2f}, "
      f"{np.percentile(noise_spreads, 95):.2f}]")
print(f"  a three-tier ranking exists in {(noise_spreads > 0).mean():.0%} of runs")
print()
print(f"the archive's {observed_spread:.2f}-point spread sits at the "
      f"{100 * (noise_spreads < observed_spread).mean():.0f}th percentile of pure noise")


         n life_satisfaction median_income        archive label
cluster                                                        
0        5              7.96       $87,424      High Well-Being
1       24              7.66       $69,004  Moderate Well-Being
2       21              6.90       $58,081     Lower Well-Being

spread between the highest and lowest cluster: 1.06 points


the same procedure on data where every true effect is zero, 600 runs:
  median spread : 0.80 points
  5th-95th      : [0.59, 1.07]
  a three-tier ranking exists in 100% of runs

the archive's 1.06-point spread sits at the 94th percentile of pure noise


The archive's clusters differ by **1.06 points** of life satisfaction between
the highest and the lowest. The same procedure on data with **no signal in it at all** produces
a spread with a median of **0.80 points** and a 5th–95th range of **0.59 to 1.07**. The
archive's separation is at the **94th percentile of pure noise** — inside the noise
distribution, near its top.

A three-tier ranking is available in **100%** of pure-noise runs, because k-means at k = 3
returns three groups with different means whatever it is given.

One further defect is worth naming because it is a coding error rather than a statistical one.
The archive assigned its labels as `{0: 'High Well-Being', 1: 'Moderate', 2: 'Lower'}` — by
**cluster index**, not by cluster mean. k-means numbers its clusters arbitrarily. The labels
happen to be monotone here; they would not have survived a change of seed, and nothing in the
code would have noticed.

## 10. The whole pipeline on nothing

The final test. Sever every true effect, so the predictors are independent of the target and
the target is noise. Then run the archived pipeline — regression, silhouette-selected
clustering, regional comparison — and count what it reports.

In [11]:
"""Run the whole pipeline on data with no signal at all."""
noise_findings = []
for seed in range(SLOW_RUNS):
    frame, _ = simulate_wellbeing(seed=seed, signal=0.0)
    fit = fit_ols(frame)
    scaled_noise = StandardScaler().fit_transform(frame[CLUSTER_FEATURES])
    best_k, best_silhouette = None, -2.0
    for k in range(2, 8):
        assignment = KMeans(n_clusters=k, random_state=ARCHIVE_SEED, n_init=10).fit_predict(
            scaled_noise
        )
        score = silhouette_score(scaled_noise, assignment)
        if score > best_silhouette:
            best_k, best_silhouette = k, score
    regional = frame.groupby("region")[TARGET].mean()
    noise_findings.append({
        "any determinant": any(fit.pvalues[k] < ALPHA for k in WELLBEING_PREDICTORS),
        "selected k": best_k,
        "silhouette": best_silhouette,
        "regional winner": regional.idxmax(),
        "regional gap": float(regional.max() - regional.min()),
    })
noise_frame = pd.DataFrame(noise_findings)

print(f"{SLOW_RUNS} datasets in which every predictor is independent of the target.")
print()
print(f"reported at least one 'determinant' at p<{ALPHA} : "
      f"{noise_frame['any determinant'].mean():.1%} of runs")
print(f"produced a silhouette-selected cluster count      : "
      f"{noise_frame['selected k'].notna().mean():.0%} of runs")
print(f"produced a highest-satisfaction region            : "
      f"{noise_frame['regional winner'].notna().mean():.0%} of runs")
print()
print("cluster count the silhouette criterion selected:")
print(noise_frame["selected k"].value_counts().sort_index().to_string())
print()
print(f"best silhouette achieved on pure noise: median "
      f"{noise_frame['silhouette'].median():.3f}, max {noise_frame['silhouette'].max():.3f}")
print(f"regional gap on pure noise: median {noise_frame['regional gap'].median():.2f} points, "
      f"95th percentile {noise_frame['regional gap'].quantile(0.95):.2f}")

archive_regional = archive.groupby("region")[TARGET].mean()
archive_gap = float(archive_regional.max() - archive_regional.min())
print()
print(f"the archive's own regional gap ({archive_regional.idxmax()} "
      f"{archive_regional.max():.2f} down to {archive_regional.idxmin()} "
      f"{archive_regional.min():.2f}) is {archive_gap:.2f} points, the "
      f"{100 * (noise_frame['regional gap'] < archive_gap).mean():.0f}th percentile of noise")


600 datasets in which every predictor is independent of the target.

reported at least one 'determinant' at p<0.05 : 19.3% of runs
produced a silhouette-selected cluster count      : 100% of runs
produced a highest-satisfaction region            : 100% of runs

cluster count the silhouette criterion selected:
selected k
2    224
3     59
4     79
5     66
6     75
7     97

best silhouette achieved on pure noise: median 0.264, max 0.351
regional gap on pure noise: median 0.28 points, 95th percentile 0.53

the archive's own regional gap (Northeast 7.55 down to South 7.21) is 0.34 points, the 69th percentile of noise


Given data containing nothing, the pipeline reports a determinant in **19.3%**
of runs, a cluster count in **100%**, and a highest-satisfaction region in **100%**.

The silhouette criterion is the one that deserves attention. It is a model-selection step, it
looks principled, and on pure noise it selects **k = 2** in 224 of 600 runs and spreads the
rest across 3 through 7 — with a best score whose median is **0.264** and whose maximum
reaches **0.351**. It never declines to choose. There is no k it can return that means
*there is no cluster structure here*.

And the archive's own regional finding — "Highest Life Satisfaction: Northeast (7.55)" — spans
**0.34 points** from top region to bottom. On pure noise that gap has a median of 0.28 and a
95th percentile of 0.53. The archive's regional result is at the **69th percentile of noise**:
not merely explainable as chance, but almost exactly the size chance produces.

## 11. Among what gets published, how much is false?

The rate in section 4 is the right number for a methodologist and the wrong one for a reader.
A reader is not looking at 2,000 runs; they are looking at a list of determinants in one
notebook, and what they want to know is what share of *that list* is false.

In [12]:
"""Among the determinants this design reports, what share are false?"""
proportions = []
for seed in range(RUNS):
    row = sweep.loc[seed]
    reported = [k for k in WELLBEING_PREDICTORS if row[f"p::{k}"] < ALPHA]
    if reported:
        proportions.append(sum(k in NULLS for k in reported) / len(reported))
proportions = np.array(proportions)

print(f"runs that reported at least one determinant: {len(proportions):,} of {RUNS:,}")
print(f"mean false-discovery proportion among reported determinants: {proportions.mean():.1%}")
print(f"runs where at least one reported determinant is false: {(proportions > 0).mean():.1%}")
print()
print("distribution of the false share within a published finding list:")
print(pd.Series(proportions).value_counts(normalize=True).sort_index().to_string(
    float_format=lambda v: f"{v:.1%}"))


runs that reported at least one determinant: 1,971 of 2,000
mean false-discovery proportion among reported determinants: 4.1%
runs where at least one reported determinant is false: 8.7%

distribution of the false share within a published finding list:
0.000000   91.3%
0.333333    2.7%
0.500000    5.5%
0.666667    0.4%
1.000000    0.2%


Across 2,000 runs, **1,971** reported at least one determinant — this design
almost always finds something. Within those published lists the mean false share is **4.1%**,
and **8.7%** of them contain at least one determinant that does not exist. In **0.2%** of runs
every reported determinant is false.

The archived notebook's list was one of the 8.7%. One of its three determinants was fictional
and it had no way to know which.

**[rule]** Report the rate that answers the reader's question. "Our threshold is 5%" describes
the procedure; "about one published finding list in twelve contains something that is not
there" describes what they are holding.

## Limitations

1. **Nothing here is a fact about well-being.** Every number in this notebook is a property of
   a generator. It supports conclusions about *methods*, and about the archived artifact, and
   about nothing else.

2. **One data-generating process.** All rates are specific to this design: four predictors, two
   of them null, n = 50, Gaussian and independent, α = 0.05. They will move under
   collinearity, heavier tails, or a different predictor count. The method of measuring them
   generalises; the numbers do not.

3. **The predictors here are independent.** Real socioeconomic predictors are strongly
   correlated, which makes coefficient recovery worse than shown, not better. Section 5's
   spread is a floor.

4. **The clipping is inherited.** `.clip()` is reproduced from the archive rather than removed,
   so the true relationship is slightly non-linear near the boundaries. This is why a
   large-sample estimate lands near, not exactly on, the declared coefficient, and the tests
   allow 6% tolerance for it.

5. **Sweep sizes are tutorial-sized.** 2,000 runs for the regression sweeps and 600 for the
   clustering ones. A rate quoted at 8.60% carries a Monte Carlo standard error near 0.6
   percentage points; treat the second digit as indicative.

6. **The archive's random forest reported 0.6136 and this notebook reproduces 0.6144.** The
   difference is a scikit-learn version change, not a data difference — every OLS figure, mean
   and cluster profile matches to the published precision. Section 8's argument does not depend
   on which of the two it is.

7. **This notebook does not claim the archive's authors were careless.** Sections 4 and 11 are
   there precisely because the opposite is true: the pipeline is conventional, the threshold is
   conventional, and the failure is what conventional practice produces at this sample size.

## Key Takeaways

1. **False discovery can only be demonstrated where truth is declared.** This is the one
   artifact in the corpus permitted to simulate, and the permission is what makes every score
   in it possible.

2. **Reproduce before you audit.** All ten of the archive's published numbers come back, which
   is what makes this an audit of that artifact rather than of a lookalike — and it cost
   keeping a legacy RNG and a dead draw.

3. **One of three published determinants had a true coefficient of exactly zero**, and the
   output gave a reader no way to tell which.

4. **Measure the design before condemning the result.** The false-positive rate is 8.60%
   against a nominal 9.75%. Nothing was done wrong; the finding was still false.

5. **The same design reports the real `health_coverage_pct` effect in only **25.6%** of runs, so it misses it roughly three times in four.** Both errors, at once, invisible
   in a single run.

6. **A coefficient without a standard error is not a finding.** Two true effects, same n, same
   model: one never wrong-signed, one wrong-signed 8.55% of the time.

7. **A ten-row test set reports which rows it got.** The published R² of 0.6144 is the 98th
   percentile across 300 splits of the same fifty rows; the median is 0.202.

8. **Procedures that cannot return nothing will not.** Feature importance gave 39.90% to
   zero-effect predictors; k-means gave a three-tier ranking in 100% of pure-noise runs; the
   silhouette criterion always picked a k.

9. **A true effect can be invisible because of its units.** The best-estimated coefficient in
   the archive printed as `0.0000`.

10. **Report the rate that answers the reader's question:** 8.7% of published finding lists
    from this design contain something that is not there.

## Exercises

1. **Find the threshold that would have saved it.** Section 3 notes `education_bachelors_pct`
   at p = 0.0743. Re-run the section 4 sweep at α = 0.10, 0.05, 0.01 and 0.001 and report both
   the false-positive rate and the detection rate for `health_coverage_pct` at each. State the
   α you would choose for this design and what it costs.

2. **Add a Bonferroni or Benjamini–Hochberg correction** to the archived regression and re-run
   the section 11 measurement. Report how much of the 8.7% it removes and how much of the
   25.6% detection rate it removes with it.

3. **Replace the ten-row test set.** Swap `train_test_split` for 5-fold cross-validation,
   repeat section 8's 300-fold procedure, and report the spread. Does it narrow enough to make
   a performance figure worth quoting at n = 50?

4. **Break the independence assumption.** Modify `simulate_wellbeing` so `median_income` and
   `education_bachelors_pct` are correlated at ρ = 0.7. Re-run sections 4 and 5. Predict the
   direction of the change before you run it, then say whether you were right.

5. **Make the cluster labels honest.** Rewrite the archive's labelling so it derives from
   cluster means rather than cluster indices, then run it across 50 seeds and show a case where
   the original hand-assigned mapping would have been backwards.

6. **Find the notebook this repository should have.** ACS table `B20017` publishes earnings by
   sex and would support a real gender-gap analysis with real margins of error. Sketch what
   sections 4 through 8 of this notebook would look like if the margins were published — what
   could be checked, and what could not.

## References

- Ioannidis, J. P. A. (2005). "Why Most Published Research Findings Are False." *PLoS
  Medicine* 2(8): e124. The positive-predictive-value framing behind section 11.
- Benjamini, Y. and Hochberg, Y. (1995). "Controlling the False Discovery Rate." *Journal of
  the Royal Statistical Society B* 57(1): 289–300.
- Gelman, A. and Carlin, J. (2014). "Beyond Power Calculations: Assessing Type S (Sign) and
  Type M (Magnitude) Errors." *Perspectives on Psychological Science* 9(6): 641–651. Sections
  5 and 6 are a worked Type-S and Type-M measurement.
- Breiman, L. (2001). "Random Forests." *Machine Learning* 45(1): 5–32, and Strobl, C. et al.
  (2007), "Bias in random forest variable importance measures," *BMC Bioinformatics* 8: 25.
- Rousseeuw, P. J. (1987). "Silhouettes: a graphical aid to the interpretation and validation
  of cluster analysis." *Journal of Computational and Applied Mathematics* 20: 53–65.
- The archived artifact under examination:
  `notebooks/09_planned/D30_subjective_wellbeing/D30_D30_subjective_wellbeing.ipynb`,
  preserved byte-for-byte in the forensic corpus.
- `src/krl_tutorials_data/simulation.py` and `tests/test_simulation.py` in this repository.
- Handbook §1.1 (published margins of error), §7.3 (precision and rank).